## t-tests and ANOVAs

- measure statistical significance in difference between means
- t-test : 2 categories
    - positive = first group has higher mean
    - negative = second group has higher mean
- ANOVA (analysis of variance) : 3 or more categories
- unlike Pearson r, they do not measure correlation
- measure if the differences in observations is statistically significant
- ANOVA : indicates difference but not which one

## finding the critical f
- critical f : value over which indicates significance
- degrees of freedom between (dfb) : categories - 1
- degrees of freedom within (dfw) : observations - categories
- alpha (significance standard) = 0.05
- critical f = stats.f.ppf(1 - alpha, dfb, dfw)
- outputs the threshold the f-value must surpass


if f is greater than critical f : do tukey hsd

In [19]:
# %conda install statsmodels

In [20]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

url = "https://raw.githubusercontent.com/dataprofessor/data/refs/heads/master/penguins_cleaned.csv"
df = pd.read_csv(url)
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181,3750,male
1,Adelie,Torgersen,39.5,17.4,186,3800,female


In [21]:
df["species"].unique()

array(['Adelie', 'Gentoo', 'Chinstrap'], dtype=object)

In [22]:
df["island"].unique()

array(['Torgersen', 'Biscoe', 'Dream'], dtype=object)

In [23]:
df["sex"].unique()

array(['male', 'female'], dtype=object)

In [24]:
males = df.loc[df["sex"] == "male", "body_mass_g"]
females = df.loc[df["sex"] == "female", "body_mass_g"]

t,p = stats.ttest_ind(males, females)
print(f"t={t:.4f}, p={p:.4f}")


t=8.5417, p=0.0000


In [25]:
deg_f = len(males) + len(females) - 2
alpha = 0.05
crit_t = stats.t.ppf(1 - alpha / 2, deg_f)
crit_t

np.float64(1.9671567996106814)

In [26]:
females.mean()

np.float64(3862.2727272727275)

In [27]:
males.mean()

np.float64(4545.684523809524)

In [28]:
print("males are bigger by this many g")
males.mean() - females.mean()

males are bigger by this many g


np.float64(683.4117965367964)

In [29]:
adelie = df.loc[df["species"] == "Adelie", "body_mass_g"]
gentoo = df.loc[df["species"] == "Gentoo", "body_mass_g"]
chinstrap = df.loc[df["species"] == "Chinstrap", "body_mass_g"]

f, p = stats.f_oneway(adelie, gentoo, chinstrap)
print(f"{f:.4f} / {p:.4f}")

341.8949 / 0.0000


In [30]:
# let's do this faster

cols = df["species"].unique()
samples = [df.loc[df["species"] == var, "body_mass_g"] for var in cols]
f,p = stats.f_oneway(*samples)
# star takes the list and unpacks it into separate arguments
print(f"f={f:.4f}, p={p:.4f}")

f=341.8949, p=0.0000


In [31]:
# critical F

dfb = len(df["species"].unique()) - 1
dfw = len(df["body_mass_g"]) - len(df["species"].unique())
alpha = 0.05
crit_f = stats.f.ppf(1 - alpha, dfb, dfw)
print(f"f={f:.4f}, threshold={crit_f:.4f}")

f=341.8949, threshold=3.0231


In [33]:
scipy_tukey = stats.tukey_hsd(*samples)
print(scipy_tukey)
cols

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)  -1386.273     0.000 -1520.255 -1252.290
 (0 - 2)    -26.924     0.916  -186.201   132.353
 (1 - 0)   1386.273     0.000  1252.290  1520.255
 (1 - 2)   1359.349     0.000  1194.430  1524.267
 (2 - 0)     26.924     0.916  -132.353   186.201
 (2 - 1)  -1359.349     0.000 -1524.267 -1194.430



array(['Adelie', 'Gentoo', 'Chinstrap'], dtype=object)

In [34]:
sm_tukey = pairwise_tukeyhsd(
    endog=df["body_mass_g"], # numeric variable
    groups=df["species"],
    alpha=0.05
)
sm_tukey.summary()

group1,group2,meandiff,p-adj,lower,upper,reject
Adelie,Chinstrap,26.9239,0.9164,-132.3528,186.2005,False
Adelie,Gentoo,1386.2726,0.0,1252.2897,1520.2554,True
Chinstrap,Gentoo,1359.3487,0.0,1194.4304,1524.2671,True
